# CS781 Phase 3 - Rhetorical Roles - ENSEMBLE + RESULT TABLES (CPU only)
Averages the sentence-level probabilities of runs (any backbone / technique) and takes the argmax per sentence.
Members are chosen on dev only (no training on dev); runs below dev 0.40 (collapsed) are left out.
Also prints the tables for the report: technique x model, seeds (noise level), design ablations x model, cost.
**Inputs:** ONE dataset with the six bundles (`bundle_A.zip`, `bundle_B.zip`, `bundle_C.zip`, `bundle_D.zip`, `bundle_E.zip`,
`bundle_extra.zip`) + `cs781-p3-code`. Runs not in the experiment table are ignored.
Internet ON, secret `HF_TOKEN`, **Accelerator: None**. Save & Run All (~3-5 min).

In [1]:
import os, sys, json, re, glob, shutil, subprocess, zipfile
W = "/kaggle/working"
WORK, P3DATA, RUNS = f"{W}/p3code", f"{W}/p3data", f"{W}/runs"
def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("rr_train.py"), WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
from huggingface_hub import hf_hub_download, list_repo_files
from kaggle_secrets import UserSecretsClient
REPO = "Exploration-Lab/CS-781-Capstone"
tok_ = UserSecretsClient().get_secret("HF_TOKEN")
os.makedirs(P3DATA, exist_ok=True)
for f in list_repo_files(REPO, repo_type="dataset", token=tok_):
    if f.startswith("Phase_3") and f.endswith(".json"):
        shutil.copy(hf_hub_download(REPO, f, repo_type="dataset", token=tok_), os.path.join(P3DATA, os.path.basename(f)))
import rr_data
for dom in ("CL", "IT"):
    json.dump({x["id"]: x["labels"] for x in rr_data.load_split(f"{P3DATA}/{dom}_dev.json")}, open(f"{W}/dev_truth_{dom}.json", "w"))
# ---- collect runs from every bundle attached (zip files, or folders Kaggle already unzipped)
os.makedirs(RUNS, exist_ok=True)
for z in glob.glob("/kaggle/input/**/bundle_*.zip", recursive=True):
    zipfile.ZipFile(z).extractall(RUNS)
for p in glob.glob("/kaggle/input/**/test_prob.npz", recursive=True):
    d = os.path.dirname(p); t = os.path.basename(d)
    if re.match(r"^[A-E]\d\d_", t) and not os.path.exists(f"{RUNS}/{t}"):
        shutil.copytree(d, f"{RUNS}/{t}")
FOUND = sorted(t for t in os.listdir(RUNS) if os.path.exists(f"{RUNS}/{t}/dev_prob.npz") and not t.startswith("ens_"))
print(len(FOUND), "runs:", FOUND)

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

125 runs: ['A01_lora', 'A02_lorma', 'A03_lorma_lr2e4', 'A04_lorma32', 'A05_rslora64', 'A06_adalora', 'A07_qlora', 'A08_ia3', 'A09_prompt', 'A10_prefix', 'A11_adapter', 'A12_bitfit', 'A13_partial', 'A14_full', 'A15_lora_abl_sentonly', 'A16_lora_abl_windowonly', 'A17_lora_abl_singlebilstm', 'A18_lora_abl_nopos', 'A19_lora_abl_noweights', 'A20_lora_abl_lastpool', 'A21_lora_abl_ctx6144', 'A22_lora_abl_bilstm2', 'A23_lora_abl_nodom', 'A24_lora_seed7', 'A25_lora_seed11', 'B01_lora', 'B02_lorma', 'B03_lorma_lr2e4', 'B04_lorma32', 'B05_rslora64', 'B06_adalora', 'B07_qlora', 'B08_ia3', 'B09_prompt', 'B10_prefix', 'B11_adapter', 'B12_bitfit', 'B13_partial', 'B14_full', 'B15_lora_abl_sentonly', 'B16_lora_abl_windowonly', 'B17_lora_abl_singlebilstm', 'B18_lora_abl_nopos', 'B19_lora_abl_noweights', 'B20_lora_abl_lastpool', 'B21_lora_abl_ctx6144', 'B22_lora_abl_bilstm2', 'B23_lora_abl_nodom', 'B24_lora_seed7', 'B25_lora_seed11', 'C01_lora', 'C02_lorma', 'C03_lorma_lr2e4', 'C04_lorma32', 'C05_rslora6

In [2]:
import numpy as np
import rr_eval as E
TEMPLATE, N_TECH, ABL, SEEDS, TAGS = ([(1, 'lora', 'LoRA r16', '-'), (2, 'lorma', 'LoRMA r16', '-'), (3, 'lorma_lr2e4', 'LoRMA r16', 'lr 2e-4'), (4, 'lorma32', 'LoRMA r32', 'rank 32'), (5, 'rslora64', 'rsLoRA r64', '-'), (6, 'adalora', 'AdaLoRA r12->8', '-'), (7, 'qlora', 'QLoRA (NF4 + LoRA r16)', '-'), (8, 'ia3', 'IA3', '-'), (9, 'prompt', 'Prompt tuning (20)', '-'), (10, 'prefix', 'Prefix tuning (20)', '-'), (11, 'adapter', 'Houlsby adapter', '-'), (12, 'bitfit', 'BitFit (norm gains)', '-'), (13, 'partial', 'Partial FT (top 4)', '-'), (14, 'full', 'Full FT', '-'), (15, 'lora_abl_sentonly', 'LoRA r16', 'sentence alone, MLP (range 1024)'), (16, 'lora_abl_windowonly', 'LoRA r16', 'packed windows, MLP head'), (17, 'lora_abl_singlebilstm', 'LoRA r16', 'sentence alone + BiLSTM (range 1536)'), (18, 'lora_abl_nopos', 'LoRA r16', 'no position features'), (19, 'lora_abl_noweights', 'LoRA r16', 'no class weights'), (20, 'lora_abl_lastpool', 'LoRA r16', 'last-token pooling'), (21, 'lora_abl_ctx6144', 'LoRA r16', 'range 6144 tokens'), (22, 'lora_abl_bilstm2', 'LoRA r16', '2-layer BiLSTM'), (23, 'lora_abl_nodom', 'LoRA r16', 'no domain embedding'), (24, 'lora_seed7', 'LoRA r16', 'seed 7'), (25, 'lora_seed11', 'LoRA r16', 'seed 11')], 14, [15, 16, 17, 18, 19, 20, 21, 22, 23], (1, 24, 25), ['A01_lora', 'A02_lorma', 'A03_lorma_lr2e4', 'A04_lorma32', 'A05_rslora64', 'A06_adalora', 'A07_qlora', 'A08_ia3', 'A09_prompt', 'A10_prefix', 'A11_adapter', 'A12_bitfit', 'A13_partial', 'A14_full', 'A15_lora_abl_sentonly', 'A16_lora_abl_windowonly', 'A17_lora_abl_singlebilstm', 'A18_lora_abl_nopos', 'A19_lora_abl_noweights', 'A20_lora_abl_lastpool', 'A21_lora_abl_ctx6144', 'A22_lora_abl_bilstm2', 'A23_lora_abl_nodom', 'A24_lora_seed7', 'A25_lora_seed11', 'B01_lora', 'B02_lorma', 'B03_lorma_lr2e4', 'B04_lorma32', 'B05_rslora64', 'B06_adalora', 'B07_qlora', 'B08_ia3', 'B09_prompt', 'B10_prefix', 'B11_adapter', 'B12_bitfit', 'B13_partial', 'B14_full', 'B15_lora_abl_sentonly', 'B16_lora_abl_windowonly', 'B17_lora_abl_singlebilstm', 'B18_lora_abl_nopos', 'B19_lora_abl_noweights', 'B20_lora_abl_lastpool', 'B21_lora_abl_ctx6144', 'B22_lora_abl_bilstm2', 'B23_lora_abl_nodom', 'B24_lora_seed7', 'B25_lora_seed11', 'C01_lora', 'C02_lorma', 'C03_lorma_lr2e4', 'C04_lorma32', 'C05_rslora64', 'C06_adalora', 'C07_qlora', 'C08_ia3', 'C09_prompt', 'C10_prefix', 'C11_adapter', 'C12_bitfit', 'C13_partial', 'C14_full', 'C15_lora_abl_sentonly', 'C16_lora_abl_windowonly', 'C17_lora_abl_singlebilstm', 'C18_lora_abl_nopos', 'C19_lora_abl_noweights', 'C20_lora_abl_lastpool', 'C21_lora_abl_ctx6144', 'C22_lora_abl_bilstm2', 'C23_lora_abl_nodom', 'C24_lora_seed7', 'C25_lora_seed11', 'D01_lora', 'D02_lorma', 'D03_lorma_lr2e4', 'D04_lorma32', 'D05_rslora64', 'D06_adalora', 'D07_qlora', 'D08_ia3', 'D09_prompt', 'D10_prefix', 'D11_adapter', 'D12_bitfit', 'D13_partial', 'D14_full', 'D15_lora_abl_sentonly', 'D16_lora_abl_windowonly', 'D17_lora_abl_singlebilstm', 'D18_lora_abl_nopos', 'D19_lora_abl_noweights', 'D20_lora_abl_lastpool', 'D21_lora_abl_ctx6144', 'D22_lora_abl_bilstm2', 'D23_lora_abl_nodom', 'D24_lora_seed7', 'D25_lora_seed11', 'E01_lora', 'E02_lorma', 'E03_lorma_lr2e4', 'E04_lorma32', 'E05_rslora64', 'E06_adalora', 'E07_qlora', 'E08_ia3', 'E09_prompt', 'E10_prefix', 'E11_adapter', 'E12_bitfit', 'E13_partial', 'E14_full', 'E15_lora_abl_sentonly', 'E16_lora_abl_windowonly', 'E17_lora_abl_singlebilstm', 'E18_lora_abl_nopos', 'E19_lora_abl_noweights', 'E20_lora_abl_lastpool', 'E21_lora_abl_ctx6144', 'E22_lora_abl_bilstm2', 'E23_lora_abl_nodom', 'E24_lora_seed7', 'E25_lora_seed11'])
ignored = [t for t in FOUND if t not in TAGS]
FOUND = [t for t in FOUND if t in TAGS]                        # only the runs of this experiment table
if ignored: print("ignored (not in the experiment table):", ignored)
def dev(t):
    return json.load(open(f"{RUNS}/{t}/summary.json"))["dev"]
def dev_f1(t):
    return dev(t)["macro_f1"]
def official(t):
    r = subprocess.run(f"python evaluate_phase3.py --truth-cl {W}/dev_truth_CL.json --truth-it {W}/dev_truth_IT.json "
                       f"--prediction {RUNS}/{t}/dev_pred.json --labels {P3DATA}/label_vocab.json", shell=True,
                       capture_output=True, text=True, cwd=WORK)
    m = re.search(r"Macro-F1:\s+([0-9.]+)", r.stdout)
    return float(m.group(1)) if m else float("nan")
RUN = {t[:3]: t for t in FOUND}                                # "D07" -> "D07_qlora"
def f1(m, n):
    t = RUN.get(f"{m}{n:02d}")
    return dev_f1(t) if t else None
def base(m):
    v = [x for x in (f1(m, n) for n in SEEDS) if x is not None]
    return v
cell = lambda v: f"{v:8.4f}" if v is not None else f"{'-':>8s}"
missing = [f"{m}{n:02d}" for m in "ABCDE" for n, *_ in TEMPLATE if f"{m}{n:02d}" not in RUN]

L = ["=== REPORT ensemble ===", f"{len(FOUND)} runs found; missing: {' '.join(missing) or 'none'}", "",
     "ALL RUNS by dev macro-F1 (official metric) | CL | IT"]
for t in sorted(FOUND, key=dev_f1, reverse=True):
    d = dev(t)
    L.append(f"{t:30s} | {d['macro_f1']:.4f} | {d['macro_f1_CL']:.4f} | {d['macro_f1_IT']:.4f}")

L.append("\nTABLE 1 - TECHNIQUE x MODEL (dev macro-F1, full design)")
L.append(f"{'':28s}" + "".join(f"{m:>8s}" for m in "ABCDE") + f"{'mean':>8s}")
for n, s, tech, ch in TEMPLATE:
    if n > N_TECH: continue
    v = [f1(m, n) for m in "ABCDE"]; ok = [x for x in v if x is not None]
    lab = tech if ch == "-" else f"{tech}, {ch}"
    L.append(f"{n:02d} {lab[:25]:25s}" + "".join(cell(x) for x in v) + cell(sum(ok) / len(ok) if ok else None))

L.append(f"\nTABLE 2 - SEEDS OF THE BASE (LoRA r16): runs {'/'.join(f'{n:02d}' for n in SEEDS)} -> mean +- std (the noise level)")
for m in "ABCDE":
    v = base(m)
    if v: L.append(f"{m}: " + " ".join(f"{x:.4f}" for x in v) + f"  -> {np.mean(v):.4f} +- {np.std(v):.4f}")

L.append("\nTABLE 3 - DESIGN ABLATIONS: change in dev macro-F1 vs the base (mean of its seeds) of the same model")
L.append(f"{'':28s}" + "".join(f"{m:>8s}" for m in "ABCDE") + f"{'mean':>8s}")
for n, s, tech, ch in TEMPLATE:
    if n not in ABL: continue
    v = [(f1(m, n) - np.mean(base(m))) if f1(m, n) is not None and base(m) else None for m in "ABCDE"]
    ok = [x for x in v if x is not None]
    L.append(f"{n:02d} {ch[:25]:25s}" + "".join(f"{x:+8.4f}" if x is not None else f"{'-':>8s}" for x in v)
             + (f"{np.mean(ok):+8.4f}" if ok else f"{'-':>8s}"))

L.append("\nTABLE 4 - COST (trainable parameters in M | peak GPU memory in GB | training minutes), techniques")
for n, s, tech, ch in TEMPLATE:
    if n > N_TECH: continue
    cells = []
    for m in "ABCDE":
        t = RUN.get(f"{m}{n:02d}")
        if t:
            sm = json.load(open(f"{RUNS}/{t}/summary.json"))
            cells.append(f"{sm['params_trainable'] / 1e6:6.2f}M {sm.get('peak_mem_gb') or 0:4.1f}G {sm.get('train_minutes') or 0:3.0f}m")
        else:
            cells.append(f"{'-':>18s}")
    lab = tech if ch == "-" else f"{tech}, {ch}"
    L.append(f"{n:02d} {lab[:25]:25s} " + " | ".join(cells))

# ---- ensembles: members chosen on dev only; collapsed runs (< 0.40) are left out
POOL = [t for t in FOUND if dev_f1(t) >= 0.40]
ranked = sorted(POOL, key=dev_f1, reverse=True)
per = {m: [t for t in ranked if t.startswith(m)] for m in "ABCDE"}
SETS = {
    "top3": ranked[:3], "top5": ranked[:5], "top10": ranked[:10], "top20": ranked[:20], "all": ranked,
    "bestpermodel": [v[0] for v in per.values() if v],
    "top3permodel": [t for v in per.values() for t in v[:3]],
    "noA_top3permodel": [t for m, v in per.items() if m != "A" for t in v[:3]],
    "techniques_all": [t for t in ranked if int(t[1:3]) <= N_TECH],
    **{f"{m}_all": v for m, v in per.items()},
}
ENS, seen = {}, set()
for name, tags in SETS.items():
    if len(tags) < 2 or tuple(sorted(tags)) in seen:
        continue
    seen.add(tuple(sorted(tags)))
    out = f"{RUNS}/ens_{name}"
    r = subprocess.run(f"python rr_ensemble.py --runs {' '.join(f'{RUNS}/{t}' for t in tags)} --data_dir {P3DATA} --out {out}",
                       shell=True, capture_output=True, text=True, cwd=WORK)
    print(f"== ens_{name} ({len(tags)}): exit {r.returncode}", r.stderr[-800:])
    if r.returncode == 0:
        ENS[f"ens_{name}"] = tags

os.makedirs(f"{W}/submissions", exist_ok=True)
L.append("\nTABLE 5 - ENSEMBLES (average of sentence probabilities) | dev macro-F1 | official | CL | IT | members")
for k, mem in sorted(ENS.items(), key=lambda kv: -json.load(open(f"{RUNS}/{kv[0]}/summary.json"))["dev"]["macro_f1"]):
    d = json.load(open(f"{RUNS}/{k}/summary.json"))["dev"]
    L.append(f"{k:22s} | {d['macro_f1']:.4f} | {official(k):.4f} | {d['macro_f1_CL']:.4f} | {d['macro_f1_IT']:.4f} | "
             f"{len(mem)}: {' '.join(mem)}")
    shutil.copy(f"{RUNS}/{k}/submission.zip", f"{W}/submissions/{k}.zip")
SHORT = {"ArgumentPetitioner": "ArgPet", "ArgumentRespondent": "ArgResp", "PrecedentReliedUpon": "PrecRel",
         "PrecedentNotReliedUpon": "PrecNRel", "PrecedentOverruled": "PrecOvr", "RulingByLowerCourt": "RulLower",
         "RatioOfTheDecision": "Ratio", "RulingByPresentCourt": "RulPres"}
L.append("\nper-label dev F1 of the ensembles")
for k in ENS:
    pl = json.load(open(f"{RUNS}/{k}/summary.json"))["dev"]["per_label"]
    L.append(f"{k:22s} " + " ".join(f"{SHORT.get(l, l)}={v['f1']:.2f}" for l, v in pl.items()))
L.append("\ntest label counts of the ensembles (Dissent / PrecedentOverruled never appear in dev)")
for k in ENS:
    c = json.load(open(f"{RUNS}/{k}/summary.json")).get("test_label_counts", {})
    L.append(f"{k:22s} Dissent={c.get('Dissent', 0)} PrecOvr={c.get('PrecedentOverruled', 0)} RulPres={c.get('RulingByPresentCourt', 0)}")
L.append("\nsubmission files: " + ", ".join(sorted(os.listdir(f"{W}/submissions"))))
rep = "\n".join(L)
open(f"{W}/REPORT_ensemble.txt", "w").write(rep)
print(rep + "\n=== END REPORT ===")

== ens_top3 (3): exit 0 
== ens_top5 (5): exit 0 
== ens_top10 (10): exit 0 
== ens_top20 (20): exit 0 
== ens_all (117): exit 0 
== ens_bestpermodel (5): exit 0 
== ens_top3permodel (15): exit 0 
== ens_noA_top3permodel (12): exit 0 
== ens_techniques_all (67): exit 0 
== ens_A_all (22): exit 0 
== ens_B_all (23): exit 0 
== ens_C_all (24): exit 0 
== ens_D_all (24): exit 0 
== ens_E_all (24): exit 0 
=== REPORT ensemble ===
125 runs found; missing: none

ALL RUNS by dev macro-F1 (official metric) | CL | IT
D23_lora_abl_nodom             | 0.5361 | 0.4368 | 0.5293
E08_ia3                        | 0.5356 | 0.4744 | 0.5312
D18_lora_abl_nopos             | 0.5329 | 0.4322 | 0.5410
D19_lora_abl_noweights         | 0.5302 | 0.4353 | 0.5339
D20_lora_abl_lastpool          | 0.5299 | 0.4169 | 0.5369
E06_adalora                    | 0.5297 | 0.4238 | 0.5326
D22_lora_abl_bilstm2           | 0.5296 | 0.4361 | 0.5416
D03_lorma_lr2e4                | 0.5288 | 0.4778 | 0.5190
D25_lora_seed11       